In [9]:
import time
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report)
from sklearn.preprocessing import StandardScaler

# Cargar los datos
ruta = r'C:\Users\jthow\iCloudDrive\Documents\3_Maestria_Estadistica_UNINORTE\3_Tercer_Semestre\Machine_Learning\tornados.csv.zip'  
df = pd.read_csv(ruta) 

# Preprocesamiento de datos
df['loss'] = df['loss'].replace(0, pd.NA)
df['loss'] = df['loss'].interpolate(method='linear')
df['mag'] = df['mag'].fillna(df['mag'].mean())
df['mortality'] = np.where(df['fat'] == 0, 0, 1)

X = df[['mag', 'slat', 'slon', 'elat', 'elon', 'len', 'wid','f1', 'f2', 'f3', 'f4','loss']]
y = df['inj']
# Dividir los datos en conjunto de entrenamiento y prueba (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [10]:
import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from statsmodels.stats.stattools import jarque_bera  # Cambio aquí
from statsmodels.stats.diagnostic import acorr_ljungbox

# Crear y ajustar el modelo SVR
svr = SVR(kernel='rbf')  # Puedes cambiar el kernel si es necesario
svr.fit(X_train, y_train)

# Predicciones del modelo SVR
y_pred_svr = svr.predict(X_test)

# Calcular las métricas
mape = mean_absolute_percentage_error(y_test, y_pred_svr)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_svr))
r2 = r2_score(y_test, y_pred_svr)

# Residuos del modelo
residuals_svr = y_test - y_pred_svr


# Jarque-Bera test
jb_test = jarque_bera(residuals_svr)

# Imprimir las métricas
print(f"SVR - MAPE: {mape:.4f}")
print(f"SVR - RMSE: {rmse:.4f}")
print(f"SVR - R2: {r2:.4f}")
print(f"Jarque-Bera Test p-value: {jb_test[1]:.4f}")

from statsmodels.stats.diagnostic import acorr_ljungbox

# Realizar el Ljung-Box test con un número de lags, por ejemplo 10
ljung_box_test = acorr_ljungbox(residuals_svr, lags=10)

# Acceder al p-valor para el último lag
ljung_box_p_value = ljung_box_test['lb_pvalue'].iloc[-1]

# Imprimir el p-valor del Ljung-Box test
print(f"Ljung-Box Test p-value: {ljung_box_p_value:.4f}")



SVR - MAPE: 503333961001455.3125
SVR - RMSE: 22.7545
SVR - R2: 0.0224
Jarque-Bera Test p-value: 0.0000
Ljung-Box Test p-value: 0.9425


# Metricas con Pipeline y Gridsearch


In [19]:
import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, train_test_split


# Definir el Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Paso 1: Escalado de los datos
    ('svr', SVR())  # Paso 2: Modelo SVR
])

# Definir los parámetros para GridSearchCV
param_grid = {
    'svr__C': [0.1, 1, 10, 100],
    'svr__epsilon': [0.01, 0.1, 0.2, 0.5],
    'svr__kernel': ['linear', 'poly', 'rbf', 'sigmoid']
}

# Configurar GridSearchCV
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_squared_error')

# Ajustar el modelo con GridSearchCV
grid_search.fit(X_train, y_train)

# Obtener el mejor modelo
best_model = grid_search.best_estimator_

# Realizar las predicciones con el mejor modelo
y_pred_svr = best_model.predict(X_test)

# Calcular las métricas
mape = mean_absolute_percentage_error(y_test, y_pred_svr)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_svr))
r2 = r2_score(y_test, y_pred_svr)

# Residuos del modelo
residuals_svr = y_test - y_pred_svr

# Realizar el Jarque-Bera test para comprobar la normalidad de los residuos
jb_test = jarque_bera(residuals_svr)

# Imprimir las métricas
print(f"SVR - MAPE: {mape:.4f}")
print(f"SVR - RMSE: {rmse:.4f}")
print(f"SVR - R2: {r2:.4f}")
print(f"Jarque-Bera Test p-value: {jb_test[1]:.4f}")

# Realizar el Ljung-Box test para comprobar autocorrelación en los residuos
ljung_box_test = acorr_ljungbox(residuals_svr, lags=[10], return_df=True)

# Acceder al p-valor para el último lag
if not ljung_box_test.empty:
    ljung_box_p_value = ljung_box_test['lb_pvalue'].iloc[-1]
    print(f"Ljung-Box Test p-value (lag 10): {ljung_box_p_value:.4f}")
else:
    print("La prueba de Ljung-Box no devolvió resultados.")

SVR - MAPE: 0.3915
SVR - RMSE: 0.6905
SVR - R2: -0.2936
Jarque-Bera Test p-value: 0.3416
Ljung-Box Test p-value (lag 10): 0.0161
